In [ ]:
%pip install langchain_community
%pip install openai
%pip install yake
%pip install transformers
%pip install nltk
%pip install torch

In [ ]:
import os
from langchain_community.llms import Ollama
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI
import pandas as pd
# from docx import Document
import csv
import json

In [ ]:
import json
import pandas as pd
import yake
from transformers import pipeline
import nltk
from nltk.stem import WordNetLemmatizer
import re

In [ ]:
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
openai.api_key = "ollama"  # Any non-empty string; not validated for local Ollama
openai.api_base = "http://localhost:11434/v1"

In [ ]:
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)  # Often needed with WordNet

# Explicit sentiment analysis model
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True  # Helps with long texts
)

def safe_sentiment(text):
    if not isinstance(text, str) or text.strip() == "":
        return {"label": "N/A", "score": 0.0}
    try:
        result = sentiment_pipeline(text, truncation=True)[0]
        return result
    except Exception as e:
        print(f"Sentiment analysis error: {e}")
        return {"label": "N/A", "score": 0.0}

In [ ]:
df = pd.read_excel("./data/raw_extracted_data.xlsx")

In [ ]:
#Fetch row and store in it a variable
row_prompts = []

for idx, row in df.iterrows():
    # Only use the 'Description' column (replace with your actual column name)
    column_value = row["Detailed Feedback"]  # <-- change to the column you want to summarize
    prompt = f"Summarize the following:\n{column_value}"
    row_prompts.append((idx, row, prompt))

In [ ]:
prompt_template = """
Carefully read the following content and please summarize the review like a short summary that captures important insights, features, observations.
Go beyond high-level abstraction — include specific details, examples, comparision with other maket tools or nuanced feedback if present. Highlight what stands out, whether it's strong opinions, repeated concerns, uncommon suggestions. Maintain a professional and coherent tone. Avoid sentiments, don't provide ratings, headings, bullet points, or lists. Avoid using word - praise, instead use highlight.

Do not include any extra title or header while returning the short summary.

Content:
{reviews}
"""

In [ ]:

# Initialize model
llm = Ollama(model="llama3.2:1b", temperature=0)

def generate_tag(summary):
    # Single-word category tag derived from the review summary
    if not isinstance(summary, str) or summary.strip() == "":
        return "N/A"
    tag_prompt = f"""Read the following review summary and respond with exactly one single word that best tags its main theme (for example: Performance, Usability, Pricing, Support, Reliability, Integration, Security, FeatureRequest, Bug, Positive, Negative). Respond with only the single word, no punctuation, no explanation.

Summary:
{summary}"""
    try:
        words = llm.invoke(tag_prompt).strip().split()
        return words[0].strip(".,!?") if words else "N/A"
    except Exception as e:
        print(f"Tagging error: {e}")
        return "N/A"

def generate_recommendation(summary):
    if not isinstance(summary, str) or summary.strip() == "":
        return "N/A"
    recommendation_prompt = f"""Read the following review summary and provide one concise, actionable AI recommendation for the product team based on it. Respond in a single sentence, no headers or bullet points.

Summary:
{summary}"""
    try:
        return llm.invoke(recommendation_prompt).strip()
    except Exception as e:
        print(f"Recommendation error: {e}")
        return "N/A"

# Prepare to collect summaries
row_summaries = []

# Iterate and collect summaries
for idx, row, prompt in row_prompts:
    reviews = row.get("Review") or row  # adjust depending on your structure
    prompt = prompt_template.format(reviews=reviews)
    feedback = row.get('Detailed Feedback', '')
    summary = llm.invoke(prompt)

    # Extract Published Date and Review Source
    published_date = ", ".join(row.get('Published Date')) if isinstance(row.get('Published Date'), list) else str(row.get('Published Date'))
    review_source = row.get('Review Source', 'Unknown')  # Default to 'Unknown' if not present
    sentiment_result = safe_sentiment(feedback)
    tag = generate_tag(summary)
    recommendation = generate_recommendation(summary)

    row_summaries.append({
        "Published Date": published_date,
        "Review Source": review_source,
        "Review Summary": summary,
        "Detailed Feedback": feedback,
        "Sentiment": sentiment_result['label'],
        "Confidence": sentiment_result['score'],
        "Tag": tag,
        "AI Recommendation": recommendation
    })

    print(f"\n🧾 Summary for row {idx + 1}:\n{summary}")


In [ ]:
# Write to CSV file
csv_file_path = "./output/genAI_Customer_Reviews_Summaries.csv"
with open(csv_file_path, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=["Published Date", "Review Source", "Review Summary",  "Detailed Feedback", "Sentiment", "Confidence", "Tag", "AI Recommendation"])
    writer.writeheader()
    writer.writerows(row_summaries)

print(f"✅ CSV file '{csv_file_path}' created successfully.")

In [ ]:
# Combine all individual review summaries into one block of text
all_summaries = "\n".join(row.get("Review Summary", "") for row in row_summaries)

# Construct the final summarization prompt with emphasis on comparison feedback
final_prompt = f"""Here are summaries of individual customer reviews:\n{all_summaries}

Based on this information, write a comprehensive final summary in paragraph form. 
Identify key patterns, trends, and insights shared across reviews. 
Also highlight any comparisons or feedback where customers referenced or compared this product/tool with other similar tools or competitors.
Avoid using bullet points or section headers; keep the summary in a well-structured paragraph."""

# Invoke the model to generate the final summary
final_summary = llm.invoke(final_prompt)

print("\nFinal Comprehensive Summary:\n")
print(final_summary)
